# *Whistleblower-as-a-Service* — caderno-demo

Demonstração reprodutível do modelo baseado em agentes (Mesa 3.x) do mecanismo
*Whistleblower-as-a-Service* (WaaS), usando o **pacote instalado** `waas_antitrust`
como fonte única — o modelo **não** é reimplementado aqui.

> **Aviso.** Este é um *demo* curto (varredura de Sobol reduzida para rodar rápido).
> Os resultados definitivos do artigo usam `n_base=1024`. Veja as limitações ao final
> e o backlog de pesquisa R01–R06 em `docs/DECISIONS.md`.

In [ ]:
# Instala o pacote apenas no Google Colab (no-op em ambiente local/CI)
import sys

if "google.colab" in sys.modules:
    !pip install --quiet "waas-antitrust @ git+https://github.com/freirelucas/waas-antitrust.git@main"


## 1. O modelo nos três regimes

Regime **A** (sem canal de denúncia), **B** (WaaS via Resolução) e **C** (WaaS via
Lei). Rodamos o modelo e comparamos detecção e bem-estar.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from waas_antitrust.model import WaaSModel, WaaSParametros
from waas_antitrust.sobol.execucao import calcular_bem_estar
from waas_antitrust.viz import PALETA, aplicar_estilo

aplicar_estilo()

series = {}
linhas = []
for regime in ["A", "B", "C"]:
    p = WaaSParametros(n_empresas=15, tam_medio_empresa=200, n_tiques=40,
                       regime=regime, seed=42)
    df = WaaSModel(p).executar()
    series[regime] = df
    vp = int(df["verdadeiros_positivos_acum"].max())
    fp = int(df["falsos_positivos_acum"].max())
    fn = int(df["falsos_negativos_acum"].max())
    custo = float(df["custo_recompensa_acum"].max())
    linhas.append({"regime": regime, "VP": vp, "FP": fp, "FN": fn,
                   "bem_estar": calcular_bem_estar(vp, fp, fn, custo, p.w_a_base)})

resumo = pd.DataFrame(linhas).set_index("regime")
resumo


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for regime in ["A", "B", "C"]:
    axes[0].plot(series[regime]["tique"], series[regime]["verdadeiros_positivos_acum"],
                 label=f"Regime {regime}", color=PALETA[regime], linewidth=2)
axes[0].set(xlabel="tique (trimestre)", ylabel="VP acumulados",
            title="Detecção acumulada por regime")
axes[0].legend()
resumo["bem_estar"].plot.bar(ax=axes[1], color=[PALETA[r] for r in resumo.index])
axes[1].set(ylabel="bem-estar", title="Bem-estar por regime", xlabel="regime")
fig.tight_layout()
plt.show()


## 2. Figura central — inversão da função-utilidade

In [ ]:
from waas_antitrust.viz import inversao

fig, _ = inversao.gerar_figura()
plt.show()


## 3. Diagrama de fase — coordenação tipo jogo global

In [ ]:
from waas_antitrust.viz import fase

fig, _ = fase.gerar_figura()
plt.show()


## 4. Sensibilidade de Sobol (replicada, versão curta)

A varredura usa replicação correta sobre seeds (a matriz inteira é avaliada por
réplica, preservando o pareamento de Saltelli) e os índices são mediados entre
réplicas. Aqui em escala reduzida.

In [ ]:
from waas_antitrust.sobol import PROBLEMA_SOBOL_8D, executar_varredura
from waas_antitrust.sobol.analise import calcular_indices_replicado

df_sobol = executar_varredura(n_base=8, regime="B", n_jobs=1,
                              n_empresas=5, n_tiques=10, n_replicas=2)
indices = calcular_indices_replicado(df_sobol, PROBLEMA_SOBOL_8D, metrica="bem_estar")
indices


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ind = indices.sort_values("ST")
ax.barh(ind["parâmetro"], ind["ST"], xerr=ind["ST_dp"], color="coral")
ax.set(xlabel="ST (Sobol de ordem total)", title="Sensibilidade do bem-estar (demo)")
fig.tight_layout()
plt.show()


## 5. Limitações e próximos passos

O modelo tem limitações rastreadas (backlog **R01–R06**): a dissuasão ainda não é
endógena, o "jogo global" usa limiares heurísticos (não o equilíbrio fechado), a
calibração contra os alvos do CADE/Dyck-Morse-Zingales não é formal e o canal de
falso reporte (R04) é uma primeira versão. As **Proposições 2 e 3 são, por ora,
conjecturas**.

- Repositório: <https://github.com/freirelucas/waas-antitrust>
- Backlog, proposições e protocolo: `docs/DECISIONS.md`, `docs/ODD.md`